In [1]:

from IPython.display import display, HTML
from google.colab import output

print("Adventure Quest initialized!")
print("Ready to create your game.")


Adventure Quest initialized!
Ready to create your game.


In [3]:

game_code = r"""
(() => {
  const old = document.getElementById("adventure-quest-colab");
  if (old) old.remove();

  document.body.insertAdjacentHTML("beforeend", `
  <div id="adventure-quest-colab">
    <style>
      #adventure-quest-colab {
        max-width:950px;margin:20px auto;padding:14px;
        background:#111c35;color:white;border-radius:16px;
        font-family:Arial,sans-serif;
      }
      #adventure-quest-colab canvas {
        width:100%;height:auto;display:block;
        border:3px solid #617da5;border-radius:12px;
        background:#8bd2ed;touch-action:none;
      }
      #adventure-quest-colab .hud {
        display:flex;flex-wrap:wrap;gap:8px;margin:12px 0;
      }
      #adventure-quest-colab .stat {
        background:#283d60;padding:9px 13px;
        border-radius:9px;font-weight:bold;font-size:14px;
      }
      #adventure-quest-colab button {
        border:0;border-radius:9px;padding:12px 15px;
        color:white;font-weight:bold;cursor:pointer;
        touch-action:none;font-size:14px;
      }
      #adventure-quest-colab button:active {
        transform:scale(.96);filter:brightness(1.3);
      }
      #adventure-quest-colab .green{background:#15945c}
      #adventure-quest-colab .blue{background:#2563eb}
      #adventure-quest-colab .orange{background:#c77916}
      #adventure-quest-colab .red{background:#bd3434}
      #adventure-quest-colab .controls {
        display:flex;flex-wrap:wrap;justify-content:center;
        gap:9px;margin-top:12px;
      }
      #adventure-quest-colab .hint {
        background:#263a5b;padding:12px;border-radius:10px;
        margin-top:12px;font-size:13px;line-height:1.8;
      }
    </style>

    <div style="display:flex;justify-content:space-between;align-items:center;gap:10px;flex-wrap:wrap">
      <div>
        <div style="font-size:25px;font-weight:900;color:#facc15">⚔️ ADVENTURE QUEST</div>
        <div style="font-size:12px;color:#b8c8e5">Explore • Collect • Fight • Survive</div>
      </div>
      <div id="aq-message" style="color:#a7f3d0;font-weight:bold">READY TO PLAY</div>
    </div>

    <div class="hud">
      <div class="stat">❤️ Lives: <span id="aq-lives">3</span></div>
      <div class="stat">🪙 Coins: <span id="aq-coins">0</span></div>
      <div class="stat">⭐ Score: <span id="aq-score">0</span></div>
      <div class="stat">🌍 Level: <span id="aq-level">1</span>/3</div>
    </div>

    <canvas id="aq-canvas" width="900" height="480" tabindex="0"></canvas>

    <div class="controls">
      <button class="green" id="aq-start">▶ START</button>
      <button class="blue" id="aq-pause">⏸ PAUSE</button>
      <button class="orange" id="aq-restart">↻ RESTART</button>
      <button class="blue" id="aq-sound">🔊 SOUND ON</button>
    </div>

    <div class="controls">
      <button class="blue" id="aq-left">⬅ LEFT</button>
      <button class="blue" id="aq-right">RIGHT ➡</button>
      <button class="green" id="aq-jump">⬆ JUMP</button>
      <button class="red" id="aq-attack">⚔ ATTACK</button>
    </div>

    <div class="hint">
      <b>🎮 HOW TO PLAY</b><br>
      ← → or A/D: Move | SPACE or ↑: Jump | X or J: Attack<br>
      Collect gold coins, defeat monsters, avoid falling, and reach the golden portal.
    </div>
  </div>
  `);

  const root = document.getElementById("adventure-quest-colab");
  const canvas = root.querySelector("#aq-canvas");
  const ctx = canvas.getContext("2d");
  const W = 900, H = 480, WORLD = 2200;
  const gravity = 0.62;

  let state = "menu";
  let level = 1, score = 0, coinCount = 0, lives = 3;
  let camera = 0, frame = 0, sound = true;
  let audio = null;
  let keys = {left:false,right:false};
  let platforms = [], coins = [], enemies = [], particles = [];
  let goal = WORLD - 100;

  const hero = {
    x:60,y:340,w:32,h:48,
    vx:0,vy:0,speed:4.5,
    onGround:false,facing:1,
    attack:0,attackHit:false,
    invincible:0
  };

  const worlds = [
    {
      name:"Green Forest",
      bg:"#79c5a2",
      platforms:[
        [0,440,450,40],[520,440,350,40],
        [940,440,400,40],[1410,440,400,40],
        [1880,440,320,40],
        [220,355,130,18],[600,355,130,18],
        [820,300,150,18],[1100,350,140,18],
        [1510,350,140,18],[1740,290,160,18],
        [2070,350,130,18]
      ],
      coins:[
        [170,405],[270,320],[390,400],[620,320],
        [700,405],[880,265],[1040,405],[1170,315],
        [1470,405],[1580,315],[1800,255],
        [1990,405],[2130,315]
      ],
      monsters:[400,790,1240,1650,2040]
    },
    {
      name:"Desert Adventure",
      bg:"#e6a65b",
      platforms:[
        [0,440,360,40],[440,440,350,40],
        [890,440,350,40],[1340,440,370,40],
        [1810,440,390,40],
        [170,340,130,18],[520,340,140,18],
        [800,280,140,18],[1050,350,130,18],
        [1450,340,140,18],[1720,280,160,18],
        [2060,340,140,18]
      ],
      coins:[
        [100,405],[220,305],[480,405],[570,305],
        [850,245],[1000,405],[1110,315],
        [1400,405],[1510,305],[1770,245],
        [1920,405],[2110,305]
      ],
      monsters:[300,680,1120,1580,2050]
    },
    {
      name:"Dark Castle",
      bg:"#7770a8",
      platforms:[
        [0,440,400,40],[480,440,330,40],
        [900,440,360,40],[1350,440,400,40],
        [1830,440,370,40],
        [180,340,140,18],[560,330,140,18],
        [820,275,150,18],[1050,345,140,18],
        [1460,330,150,18],[1740,270,150,18],
        [2020,340,150,18]
      ],
      coins:[
        [100,405],[240,305],[530,295],[740,405],
        [860,240],[1100,305],[1190,405],
        [1480,290],[1780,235],[1910,405],
        [2050,300]
      ],
      monsters:[340,700,1160,1600,2010]
    }
  ];

  function levelSetup() {
    const world = worlds[level-1];
    platforms = world.platforms.map(p=>({
      x:p[0],y:p[1],w:p[2],h:p[3]
    }));
    coins = world.coins.map(p=>({
      x:p[0],y:p[1],taken:false
    }));
    enemies = world.monsters.map((x,i)=>({
      x,y:392,w:34,h:38,
      min:Math.max(0,x-45),
      max:Math.min(WORLD-40,x+45),
      vx:i%2===0?1.2:-1.2,
      alive:true
    }));
    hero.x=60;
    hero.y=340;
    hero.vx=0;
    hero.vy=0;
    hero.attack=0;
    hero.invincible=0;
    camera=0;
    goal=WORLD-100;
  }

  function ui() {
    root.querySelector("#aq-lives").textContent=lives;
    root.querySelector("#aq-coins").textContent=coinCount;
    root.querySelector("#aq-score").textContent=score;
    root.querySelector("#aq-level").textContent=level;
    root.querySelector("#aq-message").textContent={
      menu:"READY TO PLAY",
      play:"🟢 PLAYING",
      pause:"⏸ PAUSED",
      dead:"💀 GAME OVER",
      won:"🏆 YOU WIN!"
    }[state];
  }

  function say(message) {
    root.querySelector("#aq-message").textContent=message;
    messageTimer=100;
  }
  let messageTimer=0;

  function beep(freq=500,duration=.07) {
    if(!sound)return;
    try {
      audio=audio||new(window.AudioContext||window.webkitAudioContext)();
      if(audio.state==="suspended")audio.resume();
      const osc=audio.createOscillator();
      const gain=audio.createGain();
      osc.frequency.value=freq;
      gain.gain.value=.035;
      osc.connect(gain);
      gain.connect(audio.destination);
      osc.start();
      osc.stop(audio.currentTime+duration);
    } catch(e) {}
  }

  function circle(x,y,r,color) {
    ctx.fillStyle=color;
    ctx.beginPath();
    ctx.arc(x,y,r,0,Math.PI*2);
    ctx.fill();
  }

  function rectangle(x,y,w,h,color) {
    ctx.fillStyle=color;
    ctx.fillRect(x,y,w,h);
  }

  function drawBackground() {
    const gradient=ctx.createLinearGradient(0,0,0,H);
    gradient.addColorStop(0,worlds[level-1].bg);
    gradient.addColorStop(1,"#d8edbd");
    ctx.fillStyle=gradient;
    ctx.fillRect(0,0,W,H);

    circle(770,75,37,"#fff2a6");

    for(let i=0;i<8;i++) {
      let x=((i*380-camera*.2)%1500+1500)%1500-200;
      circle(x,390,130,"#4b9b77");
      circle(x+100,400,100,"#397d65");
    }

    for(let i=0;i<6;i++) {
      let x=((i*220-camera*.1)%1100+1100)%1100;
      circle(x,90+(i%3)*35,22,"#fff");
      circle(x+22,80+(i%3)*35,27,"#fff");
      circle(x+48,92+(i%3)*35,20,"#fff");
    }
  }

  function drawWorld() {
    ctx.save();
    ctx.translate(-camera,0);

    platforms.forEach(p=>{
      rectangle(p.x,p.y,p.w,p.h,"#765035");
      rectangle(p.x,p.y,p.w,9,"#35a853");
      for(let x=p.x+8;x<p.x+p.w-5;x+=32)
        rectangle(x,p.y+15,12,5,"#a87845");
    });

    // Golden exit portal
    circle(goal,393,27,"#facc15");
    circle(goal,393,18,"#7c3aed");
    ctx.fillStyle="#fff";
    ctx.font="bold 13px Arial";
    ctx.fillText("EXIT",goal-15,330);

    // Coins
    coins.forEach(c=>{
      if(c.taken)return;
      circle(c.x,c.y,11,"#a66c06");
      circle(c.x,c.y,8,"#ffdb36");
      ctx.fillStyle="#a45a00";
      ctx.font="bold 12px Arial";
      ctx.fillText("$",c.x-4,c.y+4);
    });

    // Monsters
    enemies.forEach(e=>{
      if(!e.alive)return;
      circle(e.x+17,e.y+17,17,"#7c35b6");
      circle(e.x+17,e.y+25,12,"#a855f7");
      circle(e.x+11,e.y+14,4,"white");
      circle(e.x+23,e.y+14,4,"white");
      circle(e.x+11,e.y+14,2,"#15152d");
      circle(e.x+23,e.y+14,2,"#15152d");
      rectangle(e.x+7,e.y+31,7,7,"#4c1d95");
      rectangle(e.x+22,e.y+31,7,7,"#4c1d95");
    });

    // Hero
    if(!(hero.invincible>0 && frame%8<4)) {
      rectangle(hero.x+5,hero.y+34,9,13,"#253b93");
      rectangle(hero.x+20,hero.y+34,9,13,"#253b93");
      rectangle(hero.x+2,hero.y+44,14,5,"#422d23");
      rectangle(hero.x+19,hero.y+44,14,5,"#422d23");
      rectangle(hero.x+4,hero.y+17,27,21,"#e44736");
      circle(hero.x+17,hero.y+10,12,"#f6c28b");
      rectangle(hero.x+6,hero.y-1,22,7,"#61351f");
      circle(hero.x+21,hero.y+9,2,"#172554");
      rectangle(hero.x+4,hero.y+19,27,5,"#facc15");

      if(hero.attack>0) {
        ctx.save();
        ctx.translate(hero.x+17,hero.y+25);
        ctx.scale(hero.facing,1);
        rectangle(12,-19,32,5,"#e5efff");
        rectangle(13,-23,5,13,"#eab308");
        ctx.restore();
      }
    }
    ctx.restore();
  }

  function overlay(title,subtitle) {
    ctx.fillStyle="rgba(7,15,35,.78)";
    ctx.fillRect(0,0,W,H);
    ctx.textAlign="center";
    ctx.font="bold 42px Arial";
    ctx.fillStyle="#facc15";
    ctx.fillText(title,W/2,210);
    ctx.font="bold 18px Arial";
    ctx.fillStyle="white";
    ctx.fillText(subtitle,W/2,252);
    ctx.textAlign="left";
  }

  function render() {
    drawBackground();
    drawWorld();

    if(state==="menu")
      overlay("ADVENTURE QUEST","Press START to begin your journey!");
    if(state==="pause")
      overlay("PAUSED","Press PAUSE again to resume.");
    if(state==="dead")
      overlay("GAME OVER","Press RESTART to try again!");
    if(state==="won")
      overlay("YOU WIN!","All three worlds completed! Score: "+score);

    if(messageTimer>0 && state==="play") {
      ctx.fillStyle="rgba(12,22,40,.78)";
      ctx.fillRect(250,14,400,35);
      ctx.font="bold 15px Arial";
      ctx.textAlign="center";
      ctx.fillStyle="#fff4b8";
      ctx.fillText(root.dataset.notice||"",450,37);
      ctx.textAlign="left";
    }
  }

  function collision(a,b) {
    return a.x<b.x+b.w && a.x+a.w>b.x &&
           a.y<b.y+b.h && a.y+a.h>b.y;
  }

  function damage() {
    if(hero.invincible>0 || state!=="play")return;
    lives--;
    hero.invincible=100;
    hero.vy=-5;
    say("Ouch! You lost a life!");

    if(lives<=0) {
      state="dead";
      say("Game over!");
    }
    ui();
  }

  function attack() {
    if(state!=="play")return;
    hero.attack=15;
    hero.attackHit=false;
    beep(450);
  }

  function jump() {
    if(state==="play" && hero.onGround) {
      hero.vy=-13;
      hero.onGround=false;
      beep(550);
    }
  }

  function update() {
    if(state!=="play")return;
    frame++;

    if(hero.invincible>0)hero.invincible--;
    if(hero.attack>0)hero.attack--;
    if(messageTimer>0)messageTimer--;

    let move=(keys.right?1:0)-(keys.left?1:0);
    hero.vx=move*hero.speed;
    if(move!==0)hero.facing=move;

    // Horizontal movement
    hero.x+=hero.vx;
    hero.x=Math.max(0,Math.min(WORLD-hero.w,hero.x));

    // Gravity and vertical collision
    const oldBottom=hero.y+hero.h;
    hero.vy=Math.min(hero.vy+gravity,13);
    hero.y+=hero.vy;
    hero.onGround=false;

    platforms.forEach(p=>{
      if(hero.vy>=0 &&
         oldBottom<=p.y+5 &&
         hero.y+hero.h>=p.y &&
         hero.x+hero.w>p.x &&
         hero.x<p.x+p.w) {
        hero.y=p.y-hero.h;
        hero.vy=0;
        hero.onGround=true;
      }
    });

    // Falling into a gap
    if(hero.y>H+70) {
      damage();
      if(state==="play") {
        hero.x=Math.max(40,hero.x-100);
        hero.y=330;
        hero.vy=0;
      }
    }

    // Coin collection
    coins.forEach(c=>{
      if(!c.taken &&
         Math.abs(hero.x+16-c.x)<25 &&
         Math.abs(hero.y+22-c.y)<30) {
        c.taken=true;
        score+=10;
        coinCount++;
        root.dataset.notice="Coin collected! +10";
        messageTimer=100;
        beep(850);
      }
    });

    // Monster movement and combat
    enemies.forEach(e=>{
      if(!e.alive)return;
      e.x+=e.vx;
      if(e.x<e.min || e.x>e.max)e.vx*=-1;

      if(hero.attack>0 && !hero.attackHit) {
        const distance=Math.abs(hero.x+16-(e.x+17));
        const inFront=(e.x-hero.x)*hero.facing>-5;
        if(distance<65 && inFront && Math.abs(hero.y-e.y)<55) {
          e.alive=false;
          hero.attackHit=true;
          score+=50;
          root.dataset.notice="Monster defeated! +50";
          messageTimer=100;
          beep(300,.12);
        }
      }

      if(e.alive && collision(hero,e) && hero.invincible<=0) {
        damage();
      }
    });

    camera=Math.max(0,Math.min(WORLD-W,hero.x-W*.38));

    // Level completion
    if(hero.x>=goal-25) {
      if(level<3) {
        level++;
        levelSetup();
        root.dataset.notice="Level "+level+": "+worlds[level-1].name;
        messageTimer=100;
        beep(700,.2);
      } else {
        state="won";
        say("Adventure Champion!");
        beep(1000,.4);
      }
    }

    ui();
  }

  function loop() {
    update();
    render();
    requestAnimationFrame(loop);
  }

  function start() {
    if(state==="menu" || state==="pause") {
      state="play";
      say("Your adventure begins!");
      ui();
      beep(600);
    }
    canvas.focus();
  }

  function pause() {
    if(state==="play")state="pause";
    else if(state==="pause")state="play";
    ui();
  }

  function restart() {
    level=1;
    score=0;
    coinCount=0;
    lives=3;
    state="play";
    keys.left=false;
    keys.right=false;
    levelSetup();
    say("A new adventure begins!");
    ui();
    canvas.focus();
  }

  function bindHold(id,key) {
    const btn=root.querySelector("#"+id);
    btn.addEventListener("pointerdown",e=>{
      e.preventDefault();
      keys[key]=true;
    });
    ["pointerup","pointerleave","pointercancel"].forEach(ev=>{
      btn.addEventListener(ev,()=>keys[key]=false);
    });
  }

  bindHold("aq-left","left");
  bindHold("aq-right","right");

  root.querySelector("#aq-jump").addEventListener("pointerdown",e=>{
    e.preventDefault();jump();
  });
  root.querySelector("#aq-attack").addEventListener("pointerdown",e=>{
    e.preventDefault();attack();
  });
  root.querySelector("#aq-start").addEventListener("click",start);
  root.querySelector("#aq-pause").addEventListener("click",pause);
  root.querySelector("#aq-restart").addEventListener("click",restart);

  root.querySelector("#aq-sound").addEventListener("click",()=>{
    sound=!sound;
    root.querySelector("#aq-sound").textContent=sound?"🔊 SOUND ON":"🔇 SOUND OFF";
    if(sound)beep(600);
  });

  window.addEventListener("keydown",e=>{
    if(!root.isConnected)return;
    const k=e.key.toLowerCase();
    if(["arrowleft","arrowright","arrowup"," ","a","d","w","x","j"].includes(k))
      e.preventDefault();
    if(k==="arrowleft"||k==="a")keys.left=true;
    if(k==="arrowright"||k==="d")keys.right=true;
    if((k==="arrowup"||k==="w"||k===" ")&&!e.repeat)jump();
    if((k==="x"||k==="j")&&!e.repeat)attack();
  });

  window.addEventListener("keyup",e=>{
    const k=e.key.toLowerCase();
    if(k==="arrowleft"||k==="a")keys.left=false;
    if(k==="arrowright"||k==="d")keys.right=false;
  });

  window.addEventListener("blur",()=>{
    keys.left=false;
    keys.right=false;
  });

  levelSetup();
  ui();
  render();
  requestAnimationFrame(loop);

  return "Adventure Quest successfully loaded!";
})()
"""

output.eval_js(game_code)


'Adventure Quest successfully loaded!'